In [43]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize 
from nltk.stem import WordNetLemmatizer
import re
import spacy

In [44]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet') 
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Load Data

In [3]:
data = pd.read_csv("bbc_news.csv")
data

,Unnamed: 0,index,title,pubDate,guid,link,description
0,0,6684,Can I refuse to work?,"Wed, 10 Aug 2022 15:46:18 GMT",https://www.bbc.co.uk/news/business-62147992,https://www.bbc.co.uk/news/business-62147992?a...,With much of the UK enduring another period of...
1,1,9267,'Liz Truss the Brief?' World reacts to UK poli...,"Mon, 17 Oct 2022 11:35:12 GMT",https://www.bbc.co.uk/news/world-63285480,https://www.bbc.co.uk/news/world-63285480?at_m...,The UK's political chaos has been watched arou...
2,2,7387,Rationing energy is nothing new for off-grid c...,"Wed, 31 Aug 2022 05:20:18 GMT",https://www.bbc.co.uk/news/uk-scotland-highlan...,https://www.bbc.co.uk/news/uk-scotland-highlan...,Scoraig in the north west Highlands has long h...
3,3,767,The hunt for superyachts of sanctioned Russian...,"Tue, 22 Mar 2022 14:37:01 GMT",https://www.bbc.co.uk/news/60739336,https://www.bbc.co.uk/news/60739336?at_medium=...,"Wealthy Russians sanctioned by the US, EU and ..."
4,4,3712,Platinum Jubilee: 70 years of the Queen in 70 ...,"Wed, 01 Jun 2022 23:17:33 GMT",https://www.bbc.co.uk/news/uk-61660128,https://www.bbc.co.uk/news/uk-61660128?at_medi...,A quick look back at the Queen's 70 years on t...
...,...,...,...,...,...,...,...
995,995,13238,Dominic Raab: Third senior civil servant gives...,"Tue, 31 Jan 2023 22:39:11 GMT",https://www.bbc.co.uk/news/uk-politics-64470946,https://www.bbc.co.uk/news/uk-politics-6447094...,Deputy Prime Minister Dominic Raab is under in...
996,996,4730,Highlights: Radacanu beats Uytvanck,"Mon, 27 Jun 2022 17:54:28 GMT",https://www.bbc.co.uk/sport/av/tennis/61959705,https://www.bbc.co.uk/sport/av/tennis/61959705...,Watch US Open champion Emma Radacanu's best mo...
997,997,1871,In pictures: Mountain bikers descend snowy peak,"Sun, 17 Apr 2022 12:38:50 GMT",https://www.bbc.co.uk/news/uk-scotland-highlan...,https://www.bbc.co.uk/news/uk-scotland-highlan...,About 300 daring riders made an Easter weekend...
998,998,4346,"Companies must help cut living costs, says new...","Sat, 18 Jun 2022 02:15:07 GMT",https://www.bbc.co.uk/news/uk-politics-61842136,https://www.bbc.co.uk/news/uk-politics-6184213...,The government's new cost of living tsar says ...


In [8]:
#filter only titles
titles = pd.DataFrame(data["title"])

## Clean Data

### Lower

In [15]:
titles["lower"] = titles.apply(lambda title : title["title"].lower(), axis=1)

### Remove stop words

In [25]:
stop_words = set(stopwords.words('english'))
def remove_stop_words(s: str):
    return " ".join([c for c in s.split() if c not in stop_words])
titles["no_stopwords"] = titles["lower"].apply(lambda title: remove_stop_words(title))

### Remove punctation removal

In [28]:
titles['no_stopwords_no_punct'] = titles.apply(lambda x: re.sub(r"([^\w\s])", "", x['no_stopwords']), axis=1)

### Tokenize

In [34]:
#tokens raw
titles["tokens_raw"] = titles["title"].apply(lambda x : word_tokenize(x))

#tokens clean
titles["tokens_clean"] = titles["no_stopwords_no_punct"].apply(lambda x : word_tokenize(x))

### Lemmatizer

In [41]:
lemmatizer = WordNetLemmatizer()
titles["tokens_clean_lemmatized"] = titles["tokens_clean"].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])

## POS Tagging

In [47]:
tokens_clean_list = [token for block in titles["tokens_clean_lemmatized"] for token in block]

In [49]:
spacy_doc = nlp(" ".join(tokens_clean_list))

In [50]:
pos_df = pd.DataFrame(
    {
        "token": [token.text for token in spacy_doc],
        "pos_tag": [token.pos_ for token in spacy_doc]
    }
)

In [57]:
pos_df_counts = pos_df.groupby(["token", "pos_tag"]).size().reset_index(name = "count").sort_values(by= "count", ascending=False)
pos_df_counts

,token,pos_tag,count
30,2022,NUM,47
1162,england,PROPN,45
870,cup,PROPN,39
3056,say,VERB,37
3707,uk,PROPN,37
...,...,...,...
1559,halftime,NOUN,1
1560,hall,NOUN,1
1562,halloween,PROPN,1
1563,halted,VERB,1


In [58]:
nouns = pos_df_counts[pos_df_counts.pos_tag == "NOUN"][0:10]
nouns

,token,pos_tag,count
3840,war,NOUN,34
3948,world,NOUN,30
2136,man,NOUN,22
907,day,NOUN,21
3973,year,NOUN,20
1158,energy,NOUN,17
2847,record,NOUN,17
3935,woman,NOUN,16
1130,election,NOUN,16
3870,week,NOUN,16


In [59]:
# most common verbs
verbs = pos_df_counts[pos_df_counts.pos_tag == "VERB"][0:10]
verbs

,token,pos_tag,count
3056,say,VERB,37
3711,ukraine,VERB,22
358,beat,VERB,13
2133,make,VERB,13
3461,take,VERB,13
1651,hit,VERB,13
1459,get,VERB,13
1380,found,VERB,13
1473,give,VERB,11
3918,win,VERB,10


In [60]:
# most common adjectives
adj = pos_df_counts[pos_df_counts.pos_tag == "ADJ"][0:10]
adj

,token,pos_tag,count
2386,new,ADJ,31
3010,russian,ADJ,22
403,big,ADJ,12
1324,first,ADJ,12
1306,final,ADJ,11
2478,open,ADJ,11
1955,last,ADJ,10
3097,second,ADJ,8
2394,next,ADJ,8
3978,young,ADJ,7


## NER tagging

In [61]:
pos_df = pd.DataFrame(
    {
        "token": [token.text for token in spacy_doc.ents],
        "ner_tag": [token.label_ for token in spacy_doc.ents]
    }
)

In [69]:
ner_df_counts = pos_df.groupby(["token", "ner_tag"]).size().reset_index(name= "count").sort_values(by="count", ascending=False)

In [70]:
# most common people
people = ner_df_counts[ner_df_counts.ner_tag == "PERSON"][0:10]
people

,token,ner_tag,count
432,putin,PERSON,7
127,boris johnson,PERSON,5
106,antonio conte,PERSON,3
102,andy murray,PERSON,3
254,hodgkinson,PERSON,2
249,harry,PERSON,2
120,beth,PERSON,2
160,chris,PERSON,2
217,eu,PERSON,2
252,harry meghan,PERSON,2


In [74]:
# most common places
places = ner_df_counts.query("ner_tag == 'GPE'")[0:10]
places

,token,ner_tag,count
450,russia,GPE,10
233,france,GPE,10
339,london,GPE,9
531,uk,GPE,8
211,england,GPE,7
156,china,GPE,7
479,south africa,GPE,7
335,liverpool,GPE,7
109,australia,GPE,7
261,india,GPE,6
